# K-Nearest Neighbors:

---

## Importing Libraries:

In [1]:
import time

import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, make_scorer

## Importing Dataset:

In [2]:
data = pd.read_csv("email_spam_features.csv")

## Building Base Model:

In [3]:
X = data.drop(columns=['Category'])
y = data['Category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

In [4]:
label = LabelEncoder()

y_train = label.fit_transform(y_train)
y_test = label.transform(y_test)

In [5]:
scale= StandardScaler()

X_train = scale.fit_transform(X_train)
X_test = scale.transform(X_test)

In [6]:
knn = KNeighborsClassifier()

start_time = time.time()
knn.fit(X_train, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 0.03s


In [7]:
y_pred = knn.predict(X_test)

print(f"Accuracy: {(accuracy_score(y_test, y_pred) * 100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test, y_pred)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test, y_pred)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test, y_pred)*100):.2f}%")

Accuracy: 98.35%

Precision: 99.09%

Recall: 87.20%

F1 Score: 92.77%


In [8]:
matrix = confusion_matrix(y_test, y_pred)
matrix_df = pd.DataFrame(matrix, index=['Actual ham', 'Actual spam'], columns=['Predicted ham', 'Predicted spam'])
matrix_df

,Predicted ham,Predicted spam
Actual ham,906,1
Actual spam,16,109


## Hyper-Parameter Tuning:

In [9]:
params = {'n_neighbors' : [2, 3, 4, 5, 6, 7, 8, 9, 10],
         'weights' : ['uniform', 'distance']}

grid = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=params,
    scoring='precision',
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

Best Parameters: {'n_neighbors': 4, 'weights': 'uniform'}


In [10]:
params = {'n_neighbors' : [2, 3, 4, 5, 6, 7, 8, 9, 10],
         'weights' : ['uniform', 'distance']}

grid = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=params,
    scoring='f1',
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

Best Parameters: {'n_neighbors': 8, 'weights': 'distance'}


In [11]:
# Priority -> Precision:

Knn = KNeighborsClassifier(n_neighbors=4, weights='uniform')

start_time = time.time()
Knn.fit(X_train, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 0.04s


In [12]:
y_pred = Knn.predict(X_test)

print(f"Accuracy: {(accuracy_score(y_test, y_pred) * 100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test, y_pred)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test, y_pred)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test, y_pred)*100):.2f}%")

Accuracy: 98.35%

Precision: 100.00%

Recall: 86.40%

F1 Score: 92.70%


In [13]:
matrix = confusion_matrix(y_test, y_pred)
matrix_df = pd.DataFrame(matrix, index=['Actual ham', 'Actual spam'], columns=['Predicted ham', 'Predicted spam'])
matrix_df

,Predicted ham,Predicted spam
Actual ham,907,0
Actual spam,17,108


In [14]:
# Priority -> F1:

KNN = KNeighborsClassifier(n_neighbors=8, weights='distance')

start_time = time.time()
KNN.fit(X_train, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 0.03s


In [15]:
y_pred = KNN.predict(X_test)

print(f"Accuracy: {(accuracy_score(y_test, y_pred) * 100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test, y_pred)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test, y_pred)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test, y_pred)*100):.2f}%")

Accuracy: 98.35%

Precision: 99.09%

Recall: 87.20%

F1 Score: 92.77%


In [16]:
matrix = confusion_matrix(y_test, y_pred)
matrix_df = pd.DataFrame(matrix, index=['Actual ham', 'Actual spam'], columns=['Predicted ham', 'Predicted spam'])
matrix_df

,Predicted ham,Predicted spam
Actual ham,906,1
Actual spam,16,109


## Cross-Validation:

In [17]:
# Priority -> Precision:

preprocessing= ColumnTransformer(transformers=[
    ('scale', StandardScaler(), X.columns)
], remainder='passthrough')

model_pipeline = Pipeline(steps=[
     ('preprocessing', preprocessing),
     ('model', KNeighborsClassifier(n_neighbors=4, weights='uniform'))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=69)

scoring = {
    'accuracy' : 'accuracy',
    'precision' : make_scorer(precision_score, pos_label='spam'),
    'recall' : make_scorer(recall_score, pos_label='spam'),
    'f1' : make_scorer(f1_score, pos_label='spam')
}

result = cross_validate(
    estimator=model_pipeline,
    X=X,
    y=y,
    scoring=scoring,
    cv=cv,
    n_jobs=-1
)

metrics = ['accuracy', 'precision', 'recall', 'f1']

for i in metrics:
    score = result[f'test_{i}']
    print(f"{i.capitalize()}: {np.mean(score)*100:.2f}%\n")

Accuracy: 98.37%

Precision: 98.43%

Recall: 88.29%

F1: 93.08%



In [18]:
# Priority -> F1:

preprocessing= ColumnTransformer(transformers=[
    ('scale', StandardScaler(), X.columns)
], remainder='passthrough')

model_pipeline = Pipeline(steps=[
     ('preprocessing', preprocessing),
     ('model', KNeighborsClassifier(n_neighbors=8, weights='distance'))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=69)

scoring = {
    'accuracy' : 'accuracy',
    'precision' : make_scorer(precision_score, pos_label='spam'),
    'recall' : make_scorer(recall_score, pos_label='spam'),
    'f1' : make_scorer(f1_score, pos_label='spam')
}

result = cross_validate(
    estimator=model_pipeline,
    X=X,
    y=y,
    scoring=scoring,
    cv=cv,
    n_jobs=-1
)

metrics = ['accuracy', 'precision', 'recall', 'f1']

for i in metrics:
    score = result[f'test_{i}']
    print(f"{i.capitalize()}: {np.mean(score)*100:.2f}%\n")

Accuracy: 98.49%

Precision: 97.66%

Recall: 90.01%

F1: 93.66%

